In [86]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, precision_score

In [87]:
data = pd.read_csv('gender_classification_v7.csv')
df = data.copy()
df.head(3)

,long_hair,forehead_width_cm,forehead_height_cm,nose_wide,nose_long,lips_thin,distance_nose_to_lip_long,gender
0,1,11.8,6.1,1,0,1,1,Male
1,0,14.0,5.4,0,0,1,0,Female
2,0,11.8,6.3,1,1,1,1,Male


In [88]:
df.groupby('gender')['forehead_width_cm'].mean()

gender
Female    12.811675
Male      13.551440
Name: forehead_width_cm, dtype: float64

Буду використовувати LogisticRegression, бо дані не мають складну структуру

In [89]:
# Пропущених даних в дата-сеті немає.
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5001 entries, 0 to 5000
Data columns (total 8 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   long_hair                  5001 non-null   int64  
 1   forehead_width_cm          5001 non-null   float64
 2   forehead_height_cm         5001 non-null   float64
 3   nose_wide                  5001 non-null   int64  
 4   nose_long                  5001 non-null   int64  
 5   lips_thin                  5001 non-null   int64  
 6   distance_nose_to_lip_long  5001 non-null   int64  
 7   gender                     5001 non-null   object 
dtypes: float64(2), int64(5), object(1)
memory usage: 312.7+ KB


Треба зробити колонку "Gender" в бінарному представленні.


In [90]:
# Бачимо рівність в класах, дисбалансу в нас немає.
df['gender'].value_counts()

gender
Female    2501
Male      2500
Name: count, dtype: int64

In [91]:
# Male - 1, Female - 0.
df = pd.get_dummies(df, columns=['gender'], drop_first=True, dtype='int')
df.head()

,long_hair,forehead_width_cm,forehead_height_cm,nose_wide,nose_long,lips_thin,distance_nose_to_lip_long,gender_Male
0,1,11.8,6.1,1,0,1,1,1
1,0,14.0,5.4,0,0,1,0,0
2,0,11.8,6.3,1,1,1,1,1
3,0,14.4,6.1,0,1,1,1,1
4,1,13.5,5.9,0,0,0,0,0


In [92]:
X = df.drop('gender_Male', axis=1)
y = df['gender_Male']

In [93]:
numerical_cols = ['forehead_width_cm', 'forehead_height_cm']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols)
    ],
    remainder='passthrough'
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

cv_results = cross_val_score(pipeline, X, y, cv=skf, scoring='accuracy')

print(f"Accuracy на кожному фолді: {cv_results}")
print(f"Mean Accuracy: {cv_results.mean():.4f}")

Accuracy на кожному фолді: [0.96207585 0.948      0.966      0.966      0.976      0.954
 0.99       0.97       0.976      0.962     ]
Mean Accuracy: 0.9670
